In [0]:
# ============================================================
# NOTEBOOK : FACT_PAYMENTS
# PURPOSE  : PAYMENTS FACT LOAD
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid

In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

Max Date updated successfully


In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("payments_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.fact_payments"

    pipeline_name = "PL_FACT_PAYMENTS"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            payment_id,
            order_id,
            payment_method,
            payment_status,
            payment_amount,
            payment_date,
            transaction_reference,
            created_date,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_payments"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : payments_tbl
Rows Read : 5


FN_LOGGER LOADED SUCCESSFULLY


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            p.payment_id,

            p.order_id,

            o.customer_id,

            o.customer_name,

            upper(p.payment_method)
                AS payment_method,

            upper(p.payment_status)
                AS payment_status,

            p.payment_amount,

            p.payment_date,

            p.transaction_reference,

            CASE

                WHEN p.payment_amount >= 50000
                THEN 'HIGH_PAYMENT'

                WHEN p.payment_amount >= 10000
                THEN 'MEDIUM_PAYMENT'

                ELSE 'LOW_PAYMENT'

            END AS payment_category,

            CASE

                WHEN upper(p.payment_method) IN
                (
                    'UPI',
                    'NET_BANKING'
                )

                THEN 'ONLINE'

                ELSE 'CARD_PAYMENT'

            END AS payment_mode_group,

            p.modified_date,

            sha2(
                concat_ws(
                    '|',
                    p.order_id,
                    p.payment_method,
                    p.payment_status,
                    p.payment_amount
                ),
                256
            ) AS hash_key

        FROM vw_bronze_payments p

        LEFT JOIN silver.dim_order o
            ON p.order_id = o.order_id
            AND o.is_current = 1

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_payments"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)



Silver View Created : payments_tbl


In [0]:
# ============================================================
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE IF NOT EXISTS silver.fact_payments
        (
            payment_id BIGINT,
            order_id BIGINT,
            customer_id BIGINT,
            customer_name STRING,
            payment_method STRING,
            payment_status STRING,
            payment_amount DOUBLE,
            payment_date TIMESTAMP,
            transaction_reference STRING,
            payment_category STRING,
            payment_mode_group STRING,
            modified_date TIMESTAMP,
            hash_key STRING
        )

        USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.fact_payments


In [0]:
# ============================================================
# TRUNCATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        TRUNCATE TABLE silver.fact_payments

    """)

    print(f"Target Table Truncated : {table_name}")

except Exception as e:

    print(f"Table Truncate Failed : {table_name}")

    raise(e)

# ============================================================
# LOAD FACT TABLE
# ============================================================

try:

    spark.sql("""

        INSERT INTO silver.fact_payments

        SELECT

            payment_id,
            order_id,
            customer_id,
            customer_name,
            payment_method,
            payment_status,
            payment_amount,
            payment_date,
            transaction_reference,
            payment_category,
            payment_mode_group,
            modified_date,
            hash_key

        FROM vw_silver_payments

    """)

    print(f"Data Inserted : {table_name}")

except Exception as e:

    print(f"Fact Load Failed : {table_name}")

    raise(e)



Target Table Truncated : payments_tbl
Data Inserted : payments_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"FACT_PAYMENTS SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "FACT_PAYMENTS",
        str(e)

    )

    print(f"FACT_PAYMENTS LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : payments_tbl
Watermark Updated : payments_tbl
Audit Log Inserted : payments_tbl
FACT_PAYMENTS SUCCESSFULLY LOADED : payments_tbl
